# Chapter 07: Lattice Trajectory Planning & Quintic Splines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvknyn/self-driving-ai-course/blob/main/notebooks/07_lattice_trajectory_planning.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-181717.svg)](https://github.com/vvknyn/self-driving-ai-course)

> **The Big Question**: *How do we generate jerk-optimal, passenger-comfortable trajectories around obstacles at 70 mph?*

---

## 1. 🚨 The Real-World Dilemma
Piecewise paths require instantaneous steering changes and infinite jerk ($j(t) = \dddot{x}(t) \to \infty$), causing rollovers. Quintic ($5^{\text{th}}$-order) polynomials solve 6 boundary constraints (position, velocity, acceleration) in the Frenet frame $(s, d)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

class QuinticPolynomial:
    def __init__(self, x0, v0, a0, x1, v1, a1, T):
        self.T = T
        self.a0 = x0
        self.a1 = v0
        self.a2 = 0.5 * a0
        M = np.array([
            [   T**3,    T**4,    T**5],
            [ 3*T**2,  4*T**3,  5*T**4],
            [    6*T, 12*T**2, 20*T**3]
        ])
        b = np.array([
            x1 - (self.a0 + self.a1 * T + self.a2 * T**2),
            v1 - (self.a1 + 2 * self.a2 * T),
            a1 - (2 * self.a2)
        ])
        self.a3, self.a4, self.a5 = np.linalg.solve(M, b)

    def calc_pos(self, t):
        return self.a0 + self.a1*t + self.a2*t**2 + self.a3*t**3 + self.a4*t**4 + self.a5*t**5

    def calc_jerk(self, t):
        return 6*self.a3 + 24*self.a4*t + 60*self.a5*t**2

poly = QuinticPolynomial(x0=0.0, v0=0.0, a0=0.0, x1=3.5, v1=0.0, a1=0.0, T=3.0)
t_steps = np.linspace(0, 3.0, 100)
pos = [poly.calc_pos(t) for t in t_steps]
jerk = [poly.calc_jerk(t) for t in t_steps]

plt.figure(figsize=(9, 3.5))
plt.subplot(1, 2, 1)
plt.plot(t_steps, pos, color="#58a6ff", lw=2)
plt.title("Lateral Offset d(t) [Lane Change]")
plt.xlabel("Time (s)")
plt.ylabel("Lateral Offset (m)")
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(t_steps, jerk, color="#3fb950", lw=2)
plt.title("Jerk Profile j(t) = d3x/dt3")
plt.xlabel("Time (s)")
plt.ylabel("Jerk (m/s³)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. 🩺 Andrew Ng's Diagnostic Field Guide

| Observed Symptom | Underlying Mathematical Mechanism | Verification Test | Production Fix |
| :--- | :--- | :--- | :--- |
| **Car swerves aggressively then snaps back** | Horizon time $T$ too short, exceeding tire grip $\|a_{\text{lat}}\| > \mu g$. | Check peak lateral acceleration. | Dynamic horizon scaling: $T_{\min} \ge \sqrt{2\Delta d / a_{\text{comfort}}}$. |
| **Planner oscillates between left/right lanes** | Symmetric cost well around center obstacle. | Check cost difference $|J_L - J_R| < 10^{-3}$. | Add lane bias hysteresis. |